In [ ]:
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from typing import Literal
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
# 이 워크플로는 한 모델이 응답을 만들고(생성자), 다른 모듈이 그 응답을 평가해 피드백을 제공함. (평가자)
# 이 피드백이 '부족함'으로 판정되면 다시 생성 단계로 들어가 개선 루프를 반복함. 명확한 평가 기준이 있고, 반복 개선이 실제 품질 향상으로 이어질때 특히 효과적임.
# 글쓰기, 보고서 품질 향상, 코드 리뷰, RAG 답변 정확도 검증 등에 유용함.

# 평가자-개선자(Evaluator-Optimizer) 워크플로는 평가 기준이 명확하고, 사람이 피드백하면 개선되는 과정이 LLM에도 적용될 수 있으며, 반복을 통해 품질 향상을 측정,확인할 수 있을때 효과적임.
# 다음 예제의 목적은 한번 생성하고, 끝내는 것이 아니라 '생성 -> 평가 -> 개선' 과정을 반복해 더 나은 결과를 만들어낸 것임.

# 01. 상태 정의.
###########################################################################
# 이를 위해 먼저 워크 플로우 전반에서 주고받을 state 와 평가자가 사용할 구조회된 출력 스키마를 정의함.

# state 는 그래프 전체에서 공유되는 전역 상태로, 현재 농담(Joke), 주제(topic), 평가자가 제공한 피드백(Feedback), 그리고 농담이 재미있는지 여부(funny or not)를 포함함.
# 이 상태는 각 노드가 실행될 때 입력으로 전달되며, 노드의 실행 결과가 다시 상태에 반영함.

# 그래프 상태.
class State(TypedDict):
    joke: str # 현재 농담
    topic: str # 주제
    feedback: str # 평가자 피드백(개선 시 참고)
    funny_or_not: str # 평가 등급("funny", "not funny")

In [ ]:
# 02. 구조회된 출력 스키마
###########################################################################

# 평가자 노드에서는 자유로운 텍스트가 아니라 일정한 형식의 결과가 필요하므로 구조화된 출력 스키마인 Feedback을 정의함.
# 이 스키마는 농담이 재미있는지 여부를 나타내는 등급 과 재미없을 경우 어떻게 개선하면 좋을 지에 대한 구체적인 피드백을 함께 담도록 설계돼 있음.
# 이후 llm.with_structured_output(Feedback) 을 통해 평가자 역할의 LLM이 항상 이 구조를 따르는 결과를 반환하도록 설정함.

# 평가에 사용할 구조화 출력 스키마
class Feedback(BaseModel):
    # Literal은 정적 타입 검사기(Pyright, MyPy)에게 이 변수나 매개변수 특정 값 그자체만 가질수 있다라고 제한할 때 사용하는 타입 힌트임
    grade: Literal["funny", "not funny"] = Field(description="농담이 재미있는지(funny) 아닌지 (not funny) 판단하세요.")
    feedback = Field(discription="재미없다면 어떻게 개선할지 구체적인 피드백을 작성하세요.")

load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

# 평가자 LLM : 구조화된 출력으로 등급과 피드백을 반환.
evaluator = llm.with_structured_output(Feedback)

In [ ]:
# 03. 노드 정의
###########################################################################

# 농담을 생성하는 생성자 노드는 주제를 입력으로 받아 농담을 작성함.
# 처음 실행할 때는 단순히 주제에 맞는 농담을 생성하지만 이전 실행에서 평가자가 피드백을 남긴 경우에는 그 피드백을 함께 전달해 이전 결과를 개선한 농담을 생성함.
#이를 통해 생성자 노드는 생성 역할을 넘어서, 평가 결과를 반영한 반복 개선 역할 까지 수행함.

# 1. 생성자: 농담 생성(피드백이 있으면 반영하여 개선)
def llm_call_generator(state: State):
    if state.get("feedback"):
        msg = llm.invoke(
            f"주제: {state['topic']}\n"
            f"아래 피드백을 반영하여 더 재미있는 농담을 작성해 주세요.\n"
            f"[피드백]: {state['feedback']}"
        )
    else:
        msg = llm.invoke(f"주제: {state['topic']}\n 짧고 재치있는 농담을 작성해 주세요.")

    return {"joke": msg.content}

# 이후 평가자 노드는 생성된 농담을 입력으로 받아 그 품질을 판단함. 평가자는 농담이 충분히 재미 있는지 여부를 판단하고, 재미 없다고 판단한 경우에는 어떤 점을 개선해야 하는지에 대한 설명을 함께 반환함.
# 이 결과는 구조화된 형태로 상태에 저장되며, 이 후 실행 흐름을 결정하는 기준으로 사용됨.

# 2. 평가자: 농담 평가(등급 + 개선 피드백을 구조화하여 반환)
def llm_call_evaluator(state: State):
    grade = evaluator.invoke(
        f"다음 농담이 재미있는지 평가하고, 필요하면 개선 피드백을 주세요 \n [농담]: {state['joke']}"
    )

    return {"funny_or_not": grade.grade, "feedback": grade.feedback}

# 평가 결과를 바탕으로 실행 흐름을 제어하는 역할은 분기 함수가 담당함. 분기 함수는 평가자가 반환한 등급을 확인해 농담이 재미있다고 판단되면 워크플로우를 종료하고, 그렇지 않으면
# 생성자 노드로 다시 돌아가 개선 과정을 반복하도록 지시함.
# 이처럼 분기함수는 직접적인 작업을 수행하지 않지만, 전체 그래프의 실행 방향을 결정하는 중요한 역할을 함.

# 3. 분기: 평가 결과에 따라 종료 / 재설정 결정.
def route_joke(state: State):
    if state['funny_or_not'] == "funny":
        return "Accepted"
    elif state['funny_or_not'] == "not funny":
        return "Rejected + Feedback"

In [ ]:
# 03. 워크플로우 구성
###########################################################################

# 이제 이러한 노드들을 하나의 그래프로 연결함. 워크플로우는 시작 지점에서 생성자 노드를 실행하고, 
# 이어서 평가자 노드를 실행한 뒤, 평가 결과에 따라 종료하거나 다시 생성단계로 되돌아가는 구조로 구성함.
# 이 그래프를 컴파일한 후 한번 호출되면 농담 생성과 평가, 그리고 필요 시 반복 개선 과정이 자동으로 수행함.

# 그래프
optimizer_builder = StateGraph(State)
optimizer_builder.add_node("llm_call_generator", llm_call_generator)
optimizer_builder.add_node("llm_call_evaluator", llm_call_evaluator)

optimizer_builder.add_edge(START, "llm_call_generator")
optimizer_builder.add_edge("llm_call_generator", "llm_call_evaluator")
optimizer_builder.add_conditional_edges(
    "llm_call_evaluator",
    route_joke,
    {
        "Accepted": END,
        "Rejected + Feedback": "llm_call_generator",
    }
)

# 컴파일.
optimizer_workflow = optimizer_builder.compile()

# 다이어그램 확인
display(Image(optimizer_workflow.get_graph().draw_mermaid_png()))

# 실행
state = optimizer_workflow.invoke({"topic": "한국"})
print(state["joke"])

#실전에서는 평가 기준을 길이, 금지어, 톤, 사실성(출처 포함), 채점표 등으로 구체화해야 함.
#또한 무한 루프를 방지하기 위해 최대 반복 횟수(예: 3회)를 설정하며, 평가자의 구조화 출력을 활용해 프롬프트 주입이나 잡음에도 일관된 필드로 안전하게 파이프라인을 생성함.

#또한 필요하다면 인간 검토자(HITL, Human In The Loop)를 포함해 최종 품질을 보증할 수 있음.